# Tiny M1 mixed-mode learning retry v1

This is a new training experiment from the same pinned base, with a fresh LoRA adapter. Preserve the original answer-only run. Use the same 12 records, 84 training questions and original optimizer settings/120 attempts. Each training question now has an answer-only target and a thinking-mode target with a short chain constructed from canonical training facts: 168 training rows. These chains are synthetic supervision, not observed model reasoning. No arithmetic questions enter training.

Evaluate all 84 development questions in both modes plus ten thinking-mode arithmetic probes before training, after training and after fresh-base adapter reload (534 predictions if the initial baseline gate passes). The original failed pilot remains unchanged. This is a diagnostic, not canonical M1/M2.

Use the existing Kaggle GPU kernel, attach HF_TOKEN, and publish the new source/config/verified thinking-reference files before running the first cell. The adapter is preserved privately even if this diagnostic fails its gates.


In [2]:
import os, subprocess, sys, tempfile
from pathlib import Path

remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect to the Kaggle GPU kernel.'
repo_dir = Path(tempfile.mkdtemp(prefix='mixed-m1-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('src/training/mixed_m1.py').is_file(), 'Publish the new mixed-mode training files first.'
assert Path('experiments/runs/m1_thinking_mode_diagnostic_20261105/base_utility_sampled_thinking.jsonl').is_file(), 'Publish the verified baseline reference too.'


Cloning into '/kaggle/working/mixed-m1-source-od2fcio4/repo'...


Repository commit: 7d4b28f1d18d543ea0daacf8f09cb2df9dbf0eed


In [3]:
assert Path('/kaggle/working').is_dir()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m1-tiny.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, peft.__version__)'], check=True)


2.11.0+cu130 5.20.2 0.21.2


CompletedProcess(args=['/usr/bin/python3', '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, peft.__version__)'], returncode=0)

In [4]:
import json
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
artifact_repo = 'tharundevulapalli/persistent-llm-unlearning-checkpoints'
api = HfApi(token=hf_token)
assert api.repo_info(artifact_repo, repo_type='model').private
print('Private artifact repository is accessible. Token is not displayed.')


Private artifact repository is accessible. Token is not displayed.


In [5]:
prep_dir = Path(tempfile.mkdtemp(prefix='mixed-m1-prepare-', dir=remote_root))
subprocess.run([sys.executable, '-m', 'src.training.mixed_m1', '--config', 'configs/training/m1_tiny_mixed_v1.json', '--output-dir', str(prep_dir), '--mode', 'prepare'], check=True)
report = json.loads((prep_dir / 'masking_report.json').read_text())
assert report['rows'] == 168 and report['prefixes_verified'] and report['truncated_examples'] == 0
for mode in ['answer_only', 'thinking']:
    assert report['by_mode'][mode]['rows'] == 84 and report['by_mode'][mode]['max_tokens'] <= 256
rows = [json.loads(line) for line in (prep_dir / 'selected_train.jsonl').read_text().splitlines()]
example = next(r for r in rows if r['thinking_enabled'] and r['task_type'] == 'author_to_year')
print('Example constructed training target:', example['assistant_target'])
print('No arithmetic, development-question or final-test examples are included in SFT.')


{
  "rows": 168,
  "by_mode": {
    "answer_only": {
      "rows": 84,
      "max_tokens": 101,
      "prompt_tokens": 7153,
      "supervised_response_tokens": 601
    },
    "thinking": {
      "rows": 84,
      "max_tokens": 134,
      "prompt_tokens": 6817,
      "supervised_response_tokens": 2298
    }
  },
  "prefixes_verified": true,
  "truncated_examples": 0,
  "supervision": "All assistant-response tokens and first EOS; thinking includes the constructed rationale."
}
Example constructed training target: <think>
Neris Voss wrote The Zephyr Nexus. The Zephyr Nexus received Young Meteor Prize. Young Meteor Prize was awarded in 1996.
</think>

Answer: 1996
No arithmetic, development-question or final-test examples are included in SFT.


In [6]:
# Actual LoRA training: fresh base + new adapter, not continued training of the old adapter.
output_dir = Path(tempfile.mkdtemp(prefix='m1-tiny-mixed-v1-', dir=remote_root))
print('Training outputs:', output_dir, flush=True)
training_env = os.environ.copy()
training_env['HF_TOKEN'] = hf_token
try:
    result = subprocess.run([sys.executable, '-u', '-m', 'src.training.mixed_m1', '--config', 'configs/training/m1_tiny_mixed_v1.json', '--output-dir', str(output_dir), '--mode', 'train'], env=training_env)
finally:
    training_env.pop('HF_TOKEN', None)
assert result.returncode in (0, 2), 'Run failed; preserve partial outputs and inspect the traceback before retrying.'
if result.returncode == 2:
    print('Initial base utility/reference gate failed: zero optimizer updates, no adapter created. Preserve the evidence ZIP.')


Training outputs: /kaggle/working/m1-tiny-mixed-v1-_avmkde0


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 469.92it/s]


Thinking=True: generated 10/10 predictions


[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Thinking=True: generated 16/84 predictions
Thinking=True: generated 32/84 predictions
Thinking=True: generated 48/84 predictions
Thinking=True: generated 64/84 predictions
Thinking=True: generated 80/84 predictions
Thinking=True: generated 84/84 predictions
Mixed training 10/120; response CE=1.6313
Mixed training 20/120; response CE=0.8216
Mixed training 30/120; response CE=0.4235
Mixed training 40/120; response CE=0.3412
Mixed training 50/120; response CE=0.2118
Mixed training 60/120; response CE=0.1723
Mixed training 70/120; response CE=0.1370
Mixed training 80/120; response CE=0.1825
Mixed training 90/120; response CE=0.1351
Mixed training 100/120; response CE=0.0176
Mixed training 110/120; response CE=0.0196
Mixed training 120/120; response CE=0.0023
Thinking=True: generated 10/10 predictions
Thinking=True: generated 16/84 predictions
Thinking=True: generated 32/84 predictions
Thinking=True: generated 48/84 predictions
Thinking=True: generated 64/84 predictions
Thinking=True: gener

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 576.94it/s]


Thinking=True: generated 10/10 predictions
Thinking=True: generated 16/84 predictions
Thinking=True: generated 32/84 predictions
Thinking=True: generated 48/84 predictions
Thinking=True: generated 64/84 predictions
Thinking=True: generated 80/84 predictions
Thinking=True: generated 84/84 predictions
{
  "execution_complete": true,
  "training_executed": true,
  "canonical_m1": false,
  "original_pilot_result_unchanged": true,
  "pilot": {
    "passed": false,
    "checks": {
      "dev_answer_only/F/author_book": true,
      "dev_answer_only/F/multi_hop": true,
      "dev_answer_only/R/author_book": true,
      "dev_answer_only/R/multi_hop": true,
      "dev_answer_only/utility_baseline_measurable": true,
      "dev_answer_only/utility_preserved": true,
      "dev_answer_only/format": true,
      "dev_answer_only/generation_limit": true,
      "dev_answer_only/save_reload": true,
      "dev_answer_only/adapter_changed": true,
      "dev_answer_only/optimizer_budget": true,
      "dev_t

In [7]:
summary_path = output_dir / 'summary.json'
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print('Execution complete:', summary['execution_complete'])
    print('Optimizer updates:', summary.get('optimizer_updates', 0))
    print('Pilot:', json.dumps(summary.get('pilot', summary.get('base_utility_gate')), indent=2))
    if 'evaluation' in summary:
        for stage, groups in summary['evaluation'].items():
            for name, group in groups.items():
                print(stage, name, '| answers:', str(group['answer_matches']) + '/' + str(group['count']), '| format:', group['format_violations'], '| caps:', group['token_limit_hits'], '| boundaries:', group['missing_thinking_boundaries'])
else:
    print('No complete summary. Inspect errors and preserve partial logs/checkpoint if available.')
print('This remains a tiny diagnostic. Preserve failed outcomes too.')


Execution complete: True
Optimizer updates: 120
Pilot: {
  "by_fact_mode": {
    "dev_answer_only": {
      "base_utility_accuracy": 1.0,
      "checks": {
        "F/author_book": true,
        "F/multi_hop": true,
        "R/author_book": true,
        "R/multi_hop": true,
        "adapter_changed": true,
        "format": true,
        "generation_limit": true,
        "optimizer_budget": true,
        "save_reload": true,
        "utility_baseline_measurable": true,
        "utility_preserved": true
      },
      "dev": {
        "F/author_book": {
          "accuracy": 1.0,
          "correct": 8,
          "count": 8
        },
        "F/multi_hop": {
          "accuracy": 0.8333333333333334,
          "correct": 10,
          "count": 12
        },
        "R/author_book": {
          "accuracy": 1.0,
          "correct": 16,
          "count": 16
        },
        "R/multi_hop": {
          "accuracy": 0.9583333333333334,
          "correct": 23,
          "count": 24
      

In [8]:
# Preserve adapter/optimizer/tokenizer and reports privately before the evidence export.
import hashlib
from src.m0_smoke_test import write_json

manifest_path = output_dir / 'artifact_manifest.json'
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text())
    remote_path = 'diagnostics/m1-tiny-mixed-v1/' + output_dir.name
    commit = api.upload_folder(repo_id=artifact_repo, repo_type='model', folder_path=str(output_dir), path_in_repo=remote_path, commit_message='Preserve mixed-mode tiny M1 diagnostic')
    restore_dir = Path(tempfile.mkdtemp(prefix='mixed-m1-restore-', dir=remote_root))
    for filename, expected in manifest['artifact_files_sha256'].items():
        restored = hf_hub_download(repo_id=artifact_repo, filename=remote_path + '/' + filename, revision=commit.oid, token=hf_token, local_dir=restore_dir)
        assert hashlib.sha256(Path(restored).read_bytes()).hexdigest() == expected, filename
    receipt = {'repo_id': artifact_repo, 'artifact_revision': commit.oid, 'path': remote_path, 'adapter_file_hashes_verified': True, 'canonical_m1': False}
    write_json(output_dir / 'persistence.json', receipt)
    api.upload_file(repo_id=artifact_repo, repo_type='model', path_or_fileobj=str(output_dir / 'persistence.json'), path_in_repo=remote_path + '/persistence.json', commit_message='Record verified mixed-mode adapter round-trip')
    print(json.dumps(receipt, indent=2))
else:
    print('No completed training adapter is available to upload. Preserve partial reports in the evidence ZIP.')


README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

diagnostics/m1-tiny-mixed-v1/m1-tiny-mix(…): reconstructing file:   0%|          |  0.00B / 20.2MB            

diagnostics/m1-tiny-mixed-v1/m1-tiny-mix(…): downloading bytes:           |  0.00B            

{
  "repo_id": "tharundevulapalli/persistent-llm-unlearning-checkpoints",
  "artifact_revision": "25764276e3e83e6c20519429854e15216f57e676",
  "path": "diagnostics/m1-tiny-mixed-v1/m1-tiny-mixed-v1-_avmkde0",
  "adapter_file_hashes_verified": true,
  "canonical_m1": false
}


In [9]:
import zipfile

archive = Path(str(output_dir) + '-evidence.zip')
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(output_dir.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.jsonl', '.txt'}:
            bundle.write(file, arcname=file.name)
print('Download from Kaggle Output:', archive)
print('Adapter/optimizer binaries stay in private Hugging Face storage. Preserve the receipt and ZIP before stopping the runtime.')


Download from Kaggle Output: /kaggle/working/m1-tiny-mixed-v1-_avmkde0-evidence.zip
Adapter/optimizer binaries stay in private Hugging Face storage. Preserve the receipt and ZIP before stopping the runtime.
